In [1]:
from pathlib import Path

# Find our markdown file
md_file = list(Path('../data/pdfs').rglob('*.md'))[0]
print(f"File: {md_file.name}")

with open(md_file, 'r', encoding='utf-8') as f:
    content = f.read()

print(f"Size: {len(content):,} characters")
print(f"Lines: {len(content.splitlines()):,}")

File: Threejs_Essentials.md
Size: 249,824 characters
Lines: 6,145


In [3]:
content

'---\nsource_file: Threejs Essentials.pdf\ntitle: Three.js Essentials\nauthor: Dirksen, Jos\npages: 198\nextracted_at: 2025-11-27 14:13:00\n---\n\n# Three.js Essentials\n\n## Page 2\n\nThree.js Essentials\nCreate and animate beautiful 3D graphics with  \nthis fast-paced tutorial\nJos Dirksen\nBIRMINGHAM - MUMBAI\n\n\n\n## Page 3\n\nThree.js Essentials\nCopyright © 2014 Packt Publishing\nAll rights reserved. No part of this book may be reproduced, stored in a retrieval \nsystem, or transmitted in any form or by any means, without the prior written \npermission of the publisher, except in the case of brief quotations embedded in \ncritical articles or reviews.\nEvery effort has been made in the preparation of this book to ensure the accuracy \nof the information presented. However, the information contained in this book is \nsold without warranty, either express or implied. Neither the author, nor Packt \nPublishing, and its dealers and distributors will be held liable for any damages \n

In [5]:
pip install git+https://github.com/brandonstarxel/chunking_evaluation.git

/Users/anedu/Documents/programing/rag_pipeline/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
uv add git+https://github.com/brandonstarxel/chunking_evaluation.git

/Users/anedu/Documents/programing/rag_pipeline/.venv/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [7]:
%%capture
!pip install git+https://github.com/brandonstarxel/chunking_evaluation.git

In [8]:
from chunking_evaluation.chunking import (
    ClusterSemanticChunker,
    LLMSemanticChunker,
    FixedTokenChunker,
    RecursiveTokenChunker,
    KamradtModifiedChunker
)
# Additional Dependencies
import tiktoken
from chromadb.utils import embedding_functions
from chunking_evaluation.utils import openai_token_count
import os

ModuleNotFoundError: No module named 'chunking_evaluation'

In [9]:
def chunk_by_chars(text, size=1000, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

In [11]:
chunks = chunk_by_chars(content)

In [12]:
chunks

['---\nsource_file: Threejs Essentials.pdf\ntitle: Three.js Essentials\nauthor: Dirksen, Jos\npages: 198\nextracted_at: 2025-11-27 14:13:00\n---\n\n# Three.js Essentials\n\n## Page 2\n\nThree.js Essentials\nCreate and animate beautiful 3D graphics with  \nthis fast-paced tutorial\nJos Dirksen\nBIRMINGHAM - MUMBAI\n\n\n\n## Page 3\n\nThree.js Essentials\nCopyright © 2014 Packt Publishing\nAll rights reserved. No part of this book may be reproduced, stored in a retrieval \nsystem, or transmitted in any form or by any means, without the prior written \npermission of the publisher, except in the case of brief quotations embedded in \ncritical articles or reviews.\nEvery effort has been made in the preparation of this book to ensure the accuracy \nof the information presented. However, the information contained in this book is \nsold without warranty, either express or implied. Neither the author, nor Packt \nPublishing, and its dealers and distributors will be held liable for any damages \

In [13]:
import tiktoken

def chunk_by_tokens(text, model="gpt-3.5-turbo", chunk_size=512, overlap=50):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(text)
    
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
        start += chunk_size - overlap
    
    return chunks

In [14]:
chunks_by_tiktoken = chunk_by_tokens(content)

In [15]:
len(chunks_by_tiktoken)

128

In [16]:
from chunking_evaluation.chunking import (
    ClusterSemanticChunker,
    LLMSemanticChunker,
    FixedTokenChunker,
    RecursiveTokenChunker,
    KamradtModifiedChunker
)


/Users/anedu/Documents/programing/rag_pipeline/.venv/lib/python3.13/site-packages/chunking_evaluation/chunking/llm_semantic_chunker.py:100: SyntaxWarning: invalid escape sequence '\T'
  "Respond only with the IDs of the chunks where you believe a split should occur. YOU MUST RESPOND WITH AT LEAST ONE SPLIT. THESE SPLITS MUST BE IN ASCENDING ORDER AND EQUAL OR LARGER THAN: " + str(current_chunk)+"." + (f"\n\The previous response of {invalid_response} was invalid. DO NOT REPEAT THIS ARRAY OF NUMBERS. Please try again." if invalid_response else "")


In [17]:
def analyze_chunks(chunks, use_tokens=False):
    # Print the chunks of interest
    print("\nNumber of Chunks:", len(chunks))
    print("\n", "="*50, "200th Chunk", "="*50,"\n", chunks[199])
    print("\n", "="*50, "201st Chunk", "="*50,"\n", chunks[200])
    
    chunk1, chunk2 = chunks[199], chunks[200]
    
    if use_tokens:
        encoding = tiktoken.get_encoding("cl100k_base")
        tokens1 = encoding.encode(chunk1)
        tokens2 = encoding.encode(chunk2)
        
        # Find overlapping tokens
        for i in range(len(tokens1), 0, -1):
            if tokens1[-i:] == tokens2[:i]:
                overlap = encoding.decode(tokens1[-i:])
                print("\n", "="*50, f"\nOverlapping text ({i} tokens):", overlap)
                return
        print("\nNo token overlap found")
    else:
        # Find overlapping characters
        for i in range(min(len(chunk1), len(chunk2)), 0, -1):
            if chunk1[-i:] == chunk2[:i]:
                print("\n", "="*50, f"\nOverlapping text ({i} chars):", chunk1[-i:])
                return
        print("\nNo character overlap found")

In [18]:
def chunk_text(document, chunk_size, overlap):
    chunks = []
    stride = chunk_size - overlap
    current_idx = 0
    
    while current_idx < len(document):
        # Take chunk_size characters starting from current_idx
        chunk = document[current_idx:current_idx + chunk_size]
        if not chunk:  # Break if we're out of text
            break
        chunks.append(chunk)
        current_idx += stride  # Move forward by stride
    
    return chunks

In [19]:
character_chunks = chunk_text(content, chunk_size=400, overlap=0)

analyze_chunks(character_chunks)


Number of Chunks: 625

 ================================================== 200th Chunk ================================================== 
 the canvas as an input, is updated on each render loop.
Note that we could also do this at the end of the XMLHttpRequest 
callback for better performance, as the picture drawn on the canvas 
doesn't change after it is loaded for the first time.



## Page 62

Chapter 2
[ 49 ]
The following screenshot shows the final result:
Summary
We've seen a lot of concepts of Three.js in this chapter. The most

 ================================================== 201st Chunk ================================================== 
  important points to 
remember are as follows:
• 
We can create a good-looking earth by just starting with a basic THREE.
SphereGeometry object.
• 
Three.js provides a large number of camera controls for use. In this chapter, 
we used the OrbitControls property to quickly support the zoom and pan 
functions for our scene.
• 
There are dif

In [3]:
import sys
from pathlib import Path

In [4]:
from vectordb.chunking import (
    chunk_markdown,
    chunk_text,
    ChunkingConfig,
    ChunkingStrategy,
    get_chunk_statistics
)

ModuleNotFoundError: No module named 'vectordb'